# Lesson 04: Many market futures

> **Question.** How does the same plan perform across many possible market
> histories?
>
> **You will learn to:**
>
> 1. tell named cases from generated market paths;
> 2. build one path from trend, cycle, volatility, noise and shock components;
> 3. generate and capture market paths with a seed, and replay them;
> 4. run the plan across the paths, and read the distribution of NPV and funding
>    shortfall;
> 5. explain why a plan with no loss on the average path can still lose money on many
>    paths.
>
> **Before you start.** Lesson 03.
>
> **Time.** 60 minutes core, 30 minutes optional (use markets that move together).
>
> **Not in this lesson.** Decisions that react to the market (lesson 05).
>
> **Source.** Figure 7.2 in de Neufville and Geltner (2018), used with the
> publisher's permission and with attribution. Chapter 7 is cited for the market
> model.

## 1. The plan from lesson 03

This lesson starts from lesson 03's Model, saved as a document in the
data package. Show its equations so that nothing is hidden.

In [ ]:
import os
from datetime import date

import helpers

import rangekeeper as rk

In [ ]:
model = rk.io.load(path=helpers.data_path(name="03_base_plan.model.json"))
model.equations  # the equations from lesson 03

## 2. One market driver

Lesson 03 grew rents at a fixed rate in each case. Here, one property-wide
rent index carries the market, trend included. Add the index as a driver Flow and use
it in the income equation.

In [ ]:
yearly = rk.duration.Frequency.YEAR
years = rk.duration.Span(start=date(2027, 1, 1), end=date(2038, 1, 1)).periods(
    frequency=yearly
)
draft = model.edit()
project = draft.entity(code="project")
project.flow(key="rent_index", units="ratio", periods=years)
project.replace(key="income", equation="income = rent_index * (a_rent + b_rent)")

Lesson 03's rents compound at `rent_growth`. Now the index carries all
growth, so each rent is flat at today's income, and the Model has no rent growth to
apply. The Model cannot count growth twice, so no Specification has to remember to
set a rate to zero.

In [ ]:
project.replace(
    key="a_rent",
    equation="a_rent = rx.profile.constant(initial=a_income, start=a_open)",
)
project.replace(
    key="b_rent",
    equation="b_rent = rx.profile.constant(initial=b_income, start=b_open)",
)
project.remove(key="rent_growth")
project.remove(key="equity_irr")
model = draft.commit()

The cell also removes `equity_irr`. On a path where the equity gets nothing
back, the equity's cash flow has no change of sign, so it has no IRR, and a solve that
needs the IRR fails on that path. This lesson reads the equity NPV, which has a value
on each path.

Lesson 03's Specification fixed a rent growth rate. This Model has no
rent growth, so free that value before you move the Specification to the new
revision. `on` refuses a fixed value that the new revision does not declare.

In [ ]:
base = rk.io.load(path=helpers.data_path(name="03_base_case.specification.json"))
base = base.free("rent_growth").on(model=model)
base.roles  # lesson 03's plan, assumptions, debt and funding, by name

> **Finance note.** Market growth applied twice is a common error. If the index
> already grows 2 % a year, a 2 % rent growth on top of it gives 4 %. Put the growth
> in one place: here, the index.

## 3. A path, one component at a time

A path generator makes market paths from parameters. Start with a trend
only. Then add a cycle, then volatility and noise, then a rare shock. After each
step, **predict** the shape and then draw it. `generate` takes the name of each
series with its dynamics; here there is one series, `rent_index`.

In [ ]:
trend = rk.market.Dynamics(trend="2 % pa")
one_path = rk.market.generate(dynamics={"rent_index": trend}, periods=years, count=1)
helpers.plot_paths(paths=one_path)

In [ ]:
cycle = trend.replace(cycle_period="10 years", cycle_amplitude="10 %")
one_path = rk.market.generate(dynamics={"rent_index": cycle}, periods=years, count=1)
helpers.plot_paths(paths=one_path)

A trend and a cycle are fixed shapes. Volatility, noise and shocks are
random, so from here each path is random. A **seed** makes random paths repeatable:
the same seed and parameters give the same paths. The seed is a label for one set of
random numbers; it is not a date. `generate` refuses random dynamics without a seed.

In [ ]:
dynamics = cycle.replace(
    volatility="5 % pa", noise="2 %", shock_probability="2 % pa", shock_size="-25 %"
)
five_paths = rk.market.generate(
    dynamics={"rent_index": dynamics}, periods=years, seed=1, count=5
)
helpers.plot_paths(paths=five_paths)

> **Finance note.** Real-estate markets cycle, revert to a trend and suffer rare
> shocks. Figure 7.2 shows these components for one path.

## 4. Generate and capture

Generate many paths with one seed. The result is an `rk.market.Paths`
document. Its capture record keeps the generator, its parameters, the seed and the
library versions. Replay uses the captured values and draws no new random numbers.
A solve across many paths takes time, so the environment variable `RK_PATH_COUNT`
sets a smaller count for a quick run. `rk.io.save` never replaces a file, so the file
name has the ID of the paths: other paths go to another file.

In [ ]:
count = int(os.environ.get("RK_PATH_COUNT", 500))  # CI runs the lesson with 4 paths
paths = rk.market.generate(
    dynamics={"rent_index": dynamics}, periods=years, seed=1, count=count
)
paths.capture  # generator, parameters, seed, versions

In [ ]:
saved = f"paths/04_rent_index_{paths.id}.json"
rk.io.save(document=paths, path=saved)
assert rk.io.load(path=saved) == paths
assert paths.regenerate(seed=1) == paths  # the same seed gives the same paths

## 5. The plan on average inputs

**Predict:** value the plan on the average path. If its unlevered NPV is
above zero, how many paths lose money: none, a few or many? First, value the plan on
the average rent index.

In [ ]:
average_outcome = rk.execution.solve(
    specification=base.fix(rent_index=paths.mean(name="rent_index"))
)
average_outcome["unlevered_npv"], average_outcome["equity_npv"]

## 6. The plan across all paths

`across` makes one case for each path. A path is a set of named series.
In each case, `across` fixes each Model value from the series with the same name:
here, `rent_index`. One solve gives one result for each path.

In [ ]:
paths_outcome = rk.execution.solve(specification=base.across(paths=paths))
paths_outcome.table("unlevered_npv", "equity_npv", "peak_shortfall").head()

In [ ]:
npv = paths_outcome.distribution(key="unlevered_npv")
npv.summary()  # mean, standard deviation, 5 % and 95 % quantiles

In [ ]:
helpers.plot_target_curve(
    outcomes=paths_outcome,
    key="unlevered_npv",
    markers={"Average path": average_outcome["unlevered_npv"]},
)

## 7. The flaw of averages

Compare the result on the average path with the mean of the results
across the paths. The unlevered NPV is the same on both. Income is the rent index
times flat rents, and the NPV adds the discounted amounts of that income, so the NPV
is linear in the index. The mean of the NPVs is then the NPV on the mean path.

Leverage and the funding limit are not linear in the market, so the equity NPV, the
peak shortfall and the commitment gap can differ. On this plan they differ only
slightly. Read the difference column.

In [ ]:
rk.execution.compare(
    outcomes={"Average path": average_outcome, "Mean of paths": paths_outcome.mean()},
    keys=["unlevered_npv", "equity_npv", "peak_shortfall", "commitment_gap"],
    baseline="Average path",
)

So the mean does not show the flaw of averages on this plan. The average
path gives one result, and one result has no spread. The flaw is what that result
hides: the next section reads it.

> **Finance note.** A plan that looks good on the average path can fail on many paths.
> Read the downside: the 5 % quantile (value at risk) and the share of paths with a
> loss.

## 8. Downside and funding

Read the share of paths with a loss, first on the unlevered NPV and then
on the equity NPV. The average path shows no unlevered loss, but many paths have one.
This is the flaw of averages. Then read the share of paths with a funding shortfall.
Every path has a shortfall, and so does the average path: the loan exceeds its limit
in a good market and in a bad market. So this share cannot show the flaw. It tells
you that the plan needs more funding, whichever path occurs.

In [ ]:
equity = paths_outcome.distribution(key="equity_npv")
shortfall = paths_outcome.distribution(key="peak_shortfall")
(
    npv.probability(below="0 AUD"),
    equity.probability(below="0 AUD"),
    shortfall.probability(above="0 AUD"),
)

In [ ]:
worst = npv.case(quantile=0.05)  # the path at the 5 % quantile of the NPV
helpers.plot_flows(
    outcome=worst, keys=["equity_available", "loan_available", "committed"]
)

## 9. Optional: use markets that move together

Hotel, residential and retail rents move together, but not exactly. You
choose how. One simple structure: a market-wide index that every use shares, times a
factor of the use's own. The shared index makes the uses move together; the own
factors make them differ. `generate` makes each series independently. The structure
is yours, written as Flow arithmetic here and as an equation in the Model (lesson 07).

In [ ]:
use_paths = rk.market.generate(
    dynamics={
        "market_index": dynamics,
        "hotel_factor": rk.market.Dynamics(trend="0 % pa", volatility="6 % pa"),
        "residential_factor": rk.market.Dynamics(trend="0 % pa", volatility="3 % pa"),
        "retail_factor": rk.market.Dynamics(trend="-1 % pa", volatility="4 % pa"),
    },
    periods=years,
    seed=1,
    count=count,
)

In [ ]:
path = use_paths[use_paths.keys[0]]
helpers.plot_flows(
    flows={
        "Hotel": path["market_index"] * path["hotel_factor"],
        "Residential": path["market_index"] * path["residential_factor"],
        "Retail": path["market_index"] * path["retail_factor"],
    }
)

## Exercises

**Try it 1: more volatility.** Double the volatility. How does the 5 % quantile of
the NPV change? How does the mean change?

In [ ]:
volatile = rk.market.generate(
    dynamics={"rent_index": dynamics.replace(volatility="10 % pa")},
    periods=years,
    seed=1,
    count=count,
)
volatile_outcome = rk.execution.solve(specification=base.across(paths=volatile))
volatile_npv = volatile_outcome.distribution(key="unlevered_npv")
assert volatile_npv.quantile(level=0.05) < npv.quantile(level=0.05)

**Try it 2: growth applied twice.** In a draft, make Building A's rent compound at
2 % pa again, as in lesson 03, on top of the index. How much does the mean NPV rise?
Why is this wrong?

In [ ]:
draft = model.edit()
project = draft.entity(code="project")
project.replace(
    key="a_rent",
    equation="a_rent = rx.profile.compounding(initial=a_income, rate=2 % pa, start=a_open)",
)
twice_model = draft.commit()
twice_outcome = rk.execution.solve(
    specification=base.on(model=twice_model).across(paths=paths)
)
twice_outcome.distribution(key="unlevered_npv").mean() - npv.mean()

The mean NPV rises, but the market did not change. The index already contains the
trend, so Building A's rent now grows twice.

**Try it 3: another seed.** Regenerate the paths with seed 2. How much does
the mean NPV move? What does this say about the path count?

In [ ]:
other = paths.regenerate(seed=2)  # the same generator, periods and count
reseeded_outcome = rk.execution.solve(specification=base.across(paths=other))
reseeded_outcome.distribution(key="unlevered_npv").mean() - npv.mean()

## Checks and recap

In [ ]:
assert len(paths_outcome.cases) == count
assert paths_outcome.run.paths == paths.id  # the Run pins the paths
# The flaw of averages: no loss on the average path, but a loss on some paths.
assert average_outcome["unlevered_npv"] > rk.Quantity(text="0 AUD")
assert npv.probability(below="0 AUD") > 0
first = paths.keys[0]
assert paths_outcome.case(key=first)["rent_index"] == paths[first]["rent_index"]

**Recap.**

1. A case is a named set of assumptions. A market path is one generated history.
   Both enter a batch Specification the same way: as fixed values.
2. Market paths are captured inputs: a seed and parameters give the same paths, and
   the Run pins them.
3. The average path can show no loss when many paths lose money. Read the
   distribution, and above all its downside.

**Next.** Lesson 05 lets decisions react to the market path, using only what is
known at each date.